# Week 7 – Dimensionality Reduction: Feature Selection and PCA
## Ames Housing Dataset | 24ADI204 – Data Science & Visualization

**Syllabus mapping:** Unit 4  
**Input:** `../Week 6/Feature_Engineered_AmesHousing.csv`  
**Primary outputs:** selected features, PCA-transformed data and explained-variance metadata

### Learning objectives
1. Prepare the Week 6 feature-engineered data for dimensionality reduction.
2. Remove constant and near-constant predictors using Variance Threshold.
3. Examine target relationships and remove highly correlated predictor pairs to reduce multicollinearity.
4. Compare univariate F-regression scores with Random Forest feature importance.
5. Standardize the retained feature matrix and apply PCA.
6. Visualize individual and cumulative explained variance, a 2D PCA projection and component loadings.
7. Export selected features, PCA scores and a separate explained-variance table for Week 8 dashboard integration.

> **Method distinction:** Feature selection keeps a subset of original variables. PCA creates new orthogonal components that are combinations of the original variables. They are related but not interchangeable methods.

## 1. Import libraries and create output folders

Run this notebook after Week 6. It saves all new output files under `Week 7/` and does not overwrite the Week 6 feature-engineered dataset.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, f_regression, SelectKBest
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
RANDOM_STATE = 42
VARIANCE_THRESHOLD = 0.01
MULTICOLLINEARITY_THRESHOLD = 0.90
TOP_FEATURE_COUNT = 40

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "Week 7":
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR

input_candidates = [
    REPO_ROOT / "Week 6" / "Feature_Engineered_AmesHousing.csv",
    REPO_ROOT / "Feature_Engineered_AmesHousing.csv",
]
input_path = next((p for p in input_candidates if p.exists()), None)
if input_path is None:
    raise FileNotFoundError(
        "Week 6 feature-engineered dataset was not found. Run Week 6 first and "
        "confirm 'Week 6/Feature_Engineered_AmesHousing.csv' exists."
    )

OUTPUT_DIR = REPO_ROOT / "Week 7" / "week7_outputs"
FIGURE_DIR = REPO_ROOT / "Week 7" / "Figures"
for folder in (OUTPUT_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Input file: {input_path.relative_to(REPO_ROOT)}")
print(f"Output folder: {OUTPUT_DIR.relative_to(REPO_ROOT)}")

## 2. Load the Week 6 dataset and prepare the predictor matrix

Identifiers (`Order`, `PID`) and target columns (`SalePrice`, `SalePrice_Log`) are excluded from the PCA predictor matrix. Raw text categories are also excluded because their encoded columns are already present from Week 6.

The transformed target is used for supervised feature ranking when available. It is never included as a PCA input. A full input audit is saved before dimensionality reduction.

In [ ]:
raw_df = pd.read_csv(input_path, low_memory=False, keep_default_na=False)
# Preserve the meaningful text label "None" for structural absences, while treating blank cells as missing.
raw_df = raw_df.replace(r"^\s*$", pd.NA, regex=True)
df = raw_df.copy()
ID_COLS = [c for c in ["Order", "PID"] if c in df.columns]
TARGET_COLS = [c for c in ["SalePrice", "SalePrice_Log"] if c in df.columns]
if "SalePrice" not in df.columns:
    raise KeyError("Required target column 'SalePrice' is missing from the Week 6 dataset.")
TARGET_FOR_RANKING = "SalePrice_Log" if "SalePrice_Log" in df.columns else "SalePrice"

# Exclude raw text columns; encoded ordinal and one-hot columns remain available.
raw_text_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
exclude_cols = set(ID_COLS + TARGET_COLS + raw_text_cols)
feature_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in exclude_cols]

X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(df[TARGET_FOR_RANKING], errors="coerce")
valid_target = y.notna()
X = X.loc[valid_target].copy()
y = y.loc[valid_target].copy()

# Replace infinities and impute any residual feature gaps with the feature median.
X = X.replace([np.inf, -np.inf], np.nan)
all_missing = X.columns[X.isna().all()].tolist()
if all_missing:
    X = X.drop(columns=all_missing)
    feature_cols = [c for c in feature_cols if c not in all_missing]
for col in X.columns:
    if X[col].isna().any():
        median = X[col].median()
        X[col] = X[col].fillna(median if pd.notna(median) else 0)

input_audit = pd.DataFrame({
    "Metric": ["Rows in source file", "Columns in source file", "Raw text columns excluded", "Identifier columns excluded", "Target used for supervised ranking", "Numeric predictor columns before filtering", "Feature columns with all missing values dropped", "Missing values in predictor matrix after preparation"],
    "Value": [len(df), df.shape[1], len(raw_text_cols), len(ID_COLS), TARGET_FOR_RANKING, X.shape[1], len(all_missing), int(X.isna().sum().sum())]
})
display(input_audit)
input_audit.to_csv(OUTPUT_DIR / "week7_input_audit.csv", index=False)
print(f"Predictor matrix shape: {X.shape}")

## 3. Feature selection: Variance Threshold

A predictor with zero or extremely low variance provides little information because it has nearly the same value for every property. `VarianceThreshold(0.01)` removes constant and near-constant features from this dataset.

The threshold is applied to the numeric predictor matrix only. The removed and retained feature names are saved for review.

In [ ]:
if X.shape[1] < 2:
    raise ValueError("At least two numerical predictor columns are required for this Week 7 analysis.")

variance_filter = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
X_vt_array = variance_filter.fit_transform(X)
vt_support = variance_filter.get_support()
vt_kept = X.columns[vt_support].tolist()
vt_dropped = X.columns[~vt_support].tolist()
X_vt = pd.DataFrame(X_vt_array, columns=vt_kept, index=X.index)

variance_summary = pd.DataFrame({
    "Metric": ["Predictors before threshold", "Predictors retained", "Predictors removed", "Threshold"],
    "Value": [X.shape[1], len(vt_kept), len(vt_dropped), VARIANCE_THRESHOLD]
})
display(variance_summary)
if vt_dropped:
    display(pd.DataFrame({"Removed near-constant feature": vt_dropped}))
variance_summary.to_csv(OUTPUT_DIR / "week7_variance_threshold_summary.csv", index=False)
pd.DataFrame({"Removed feature": vt_dropped}).to_csv(OUTPUT_DIR / "week7_variance_threshold_removed.csv", index=False)

## 4. Correlation analysis and multicollinearity pruning

Two different relationships are examined:

1. **Feature-to-target correlation:** ranks predictors by their absolute Pearson correlation with the target. This is a supervised ranking aid.
2. **Feature-to-feature correlation:** identifies predictors that are highly correlated with one another. Keeping many near-duplicate predictors can make a model harder to interpret and can distort some analyses.

The greedy pruning procedure sorts features by absolute target correlation, keeps the strongest candidate, and removes a later feature when its absolute correlation with an already-kept feature is at least 0.90. This gives a reproducible representative from each highly correlated group.

> Correlation is an association measure, not proof of causation. The pruning threshold is a practical screening choice and should be checked against the saved table of removed pairs.

In [ ]:
# Save a compact feature-to-target correlation ranking.
target_corr = X_vt.apply(lambda col: col.corr(y)).replace([np.inf, -np.inf], np.nan).fillna(0)
target_corr = target_corr.sort_values(key=lambda s: s.abs(), ascending=False)
target_corr_df = target_corr.rename("Pearson correlation with target").reset_index().rename(columns={"index": "Feature"})
target_corr_df.to_csv(OUTPUT_DIR / "week7_feature_target_correlations.csv", index=False)
display(target_corr_df.head(20).round(4))

# Full pairwise correlation matrix for the variance-filtered predictors.
pair_corr = X_vt.corr().abs().fillna(0)
pair_corr.to_csv(OUTPUT_DIR / "week7_pairwise_absolute_correlation_matrix.csv")

ordered_features = target_corr.index.tolist()
kept_features = []
dropped_pairs = []
for feature in ordered_features:
    if not kept_features:
        kept_features.append(feature)
        continue
    correlations_to_kept = pair_corr.loc[feature, kept_features]
    max_corr = float(correlations_to_kept.max())
    representative = str(correlations_to_kept.idxmax())
    if max_corr >= MULTICOLLINEARITY_THRESHOLD:
        dropped_pairs.append({
            "Dropped feature": feature,
            "Retained correlated feature": representative,
            "Absolute pairwise correlation": max_corr,
            "Absolute target correlation (dropped)": abs(float(target_corr.loc[feature])),
            "Absolute target correlation (retained)": abs(float(target_corr.loc[representative])),
        })
    else:
        kept_features.append(feature)

X_reduced = X_vt[kept_features].copy()
dropped_pair_df = pd.DataFrame(dropped_pairs, columns=[
    "Dropped feature", "Retained correlated feature", "Absolute pairwise correlation",
    "Absolute target correlation (dropped)", "Absolute target correlation (retained)"
])
dropped_pair_df.to_csv(OUTPUT_DIR / "week7_multicollinearity_removed_pairs.csv", index=False)

multicollinearity_summary = pd.DataFrame({
    "Metric": ["Features after variance threshold", "Multicollinearity threshold", "Features retained after pairwise pruning", "Features removed due to pairwise correlation"],
    "Value": [len(vt_kept), MULTICOLLINEARITY_THRESHOLD, len(kept_features), len(dropped_pairs)]
})
display(multicollinearity_summary)
if not dropped_pair_df.empty:
    display(dropped_pair_df.sort_values("Absolute pairwise correlation", ascending=False).head(20).round(4))
multicollinearity_summary.to_csv(OUTPUT_DIR / "week7_multicollinearity_summary.csv", index=False)

if X_reduced.shape[1] < 2:
    raise ValueError("Pairwise correlation pruning left fewer than two predictors. Review the threshold and feature matrix.")
print(f"Predictor matrix after correlation pruning: {X_reduced.shape}")

## 5. Compare univariate F-regression and Random Forest importance

The two methods provide different views of feature relevance:

- **F-regression (`SelectKBest`)** scores the strength of each predictor's individual linear relationship with the continuous target.
- **Random Forest importance** measures how much each feature contributes to tree splits in an ensemble model and can capture nonlinear patterns.

This is an exploratory feature-ranking exercise, not a validated predictive model. The importance values should not be interpreted as causal effects.

In [ ]:
K_FEATURES = min(TOP_FEATURE_COUNT, X_reduced.shape[1])
selector = SelectKBest(score_func=f_regression, k=K_FEATURES)
selector.fit(X_reduced, y)
f_scores = pd.Series(selector.scores_, index=X_reduced.columns).replace([np.inf, -np.inf], np.nan).fillna(0)
f_rank = f_scores.rank(ascending=False, method="average")

rf = RandomForestRegressor(
    n_estimators=160,
    max_depth=14,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_reduced, y)
rf_importance = pd.Series(rf.feature_importances_, index=X_reduced.columns)
rf_rank = rf_importance.rank(ascending=False, method="average")

# Rank aggregation gives both methods equal weight without adding scores on different scales.
rank_table = pd.DataFrame({
    "Feature": X_reduced.columns,
    "F-regression score": f_scores.values,
    "F-regression rank": f_rank.values,
    "Random Forest importance": rf_importance.values,
    "Random Forest rank": rf_rank.values,
})
rank_table["Mean rank"] = rank_table[["F-regression rank", "Random Forest rank"]].mean(axis=1)
rank_table = rank_table.sort_values(["Mean rank", "Feature"]).reset_index(drop=True)
rank_table["Combined rank"] = np.arange(1, len(rank_table) + 1)
rank_table.to_csv(OUTPUT_DIR / "week7_feature_selection_scores.csv", index=False)
display(rank_table.head(20).round(5))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_rf = rf_importance.sort_values(ascending=False).head(15).sort_values()
axes[0].barh(top_rf.index, top_rf.values)
axes[0].set_title("Top 15 Random Forest feature importances")
axes[0].set_xlabel("Importance")
top_f = f_scores.sort_values(ascending=False).head(15).sort_values()
axes[1].barh(top_f.index, top_f.values)
axes[1].set_title("Top 15 F-regression scores")
axes[1].set_xlabel("F score")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "Feature_Selection_Importance.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

selected_features = rank_table.head(min(TOP_FEATURE_COUNT, len(rank_table)))["Feature"].tolist()
metadata_cols = [c for c in ID_COLS + TARGET_COLS if c in df.columns]
selected_export = pd.concat([df.loc[X_reduced.index, metadata_cols].reset_index(drop=True), X_reduced[selected_features].reset_index(drop=True)], axis=1)
selected_path = REPO_ROOT / "Week 7" / "Selected_Features_AmesHousing.csv"
selected_export.to_csv(selected_path, index=False)
print(f"Selected {len(selected_features)} features using combined F-regression and Random Forest ranks.")
print(f"Selected-feature export: {selected_path.relative_to(REPO_ROOT)}")

## 6. Standardize the reduced predictor matrix

PCA is affected by feature scale. Variables measured in thousands (for example, area) can dominate variables measured on smaller numeric scales if we do not standardize them first.

The Week 6 main export is intentionally kept unscaled. Here we apply `StandardScaler` to the retained predictors after variance and multicollinearity pruning. Identifiers and target values are not included.

In [ ]:
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X_reduced)
X_scaled = pd.DataFrame(X_scaled_array, columns=kept_features, index=X_reduced.index)

standardization_check = pd.DataFrame({
    "Metric": ["Mean across standardized feature matrix", "Standard deviation across standardized feature matrix", "Number of predictors entering PCA"],
    "Value": [float(X_scaled.to_numpy().mean()), float(X_scaled.to_numpy().std()), X_scaled.shape[1]]
})
display(standardization_check.round(6))
standardization_check.to_csv(OUTPUT_DIR / "week7_standardization_check.csv", index=False)

## 7. Principal Component Analysis and explained variance

PCA transforms correlated standardized predictors into new, mutually orthogonal components. Each component explains a proportion of the total variance.

We calculate the number of components needed to retain at least 80%, 90% and 95% cumulative variance. The notebook exports **all** individual explained-variance ratios and their cumulative sums to `PCA_Explained_Variance.csv`. Week 8's dashboard is configured to look for this filename.

In [ ]:
pca_full = PCA(svd_solver="full")
pca_full.fit(X_scaled)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

def components_for_threshold(cumulative_values, threshold):
    hits = np.flatnonzero(cumulative_values >= threshold)
    return int(hits[0] + 1) if len(hits) else len(cumulative_values)

n_80 = components_for_threshold(cumulative, 0.80)
n_90 = components_for_threshold(cumulative, 0.90)
n_95 = components_for_threshold(cumulative, 0.95)

variance_df = pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(len(explained))],
    "Component number": np.arange(1, len(explained) + 1),
    "Explained Variance Ratio": explained,
    "Cumulative Explained Variance": cumulative,
})
variance_path = REPO_ROOT / "Week 7" / "PCA_Explained_Variance.csv"
variance_df.to_csv(variance_path, index=False)

pca_summary = pd.DataFrame({
    "Variance target": ["80%", "90%", "95%"],
    "Components required": [n_80, n_90, n_95],
    "Original predictors entering PCA": [X_scaled.shape[1]] * 3,
    "Dimensionality reduction (%)": [round((1 - n / X_scaled.shape[1]) * 100, 2) for n in [n_80, n_90, n_95]],
})
display(pca_summary)
print(f"PC1 explained variance: {explained[0]*100:.3f}%")
print(f"PC2 explained variance: {explained[1]*100:.3f}%" if len(explained) > 1 else "Only one component is available.")
print(f"Components for 95% variance: {n_95}")
pca_summary.to_csv(OUTPUT_DIR / "week7_pca_component_summary.csv", index=False)
print(f"Saved Week 8 variance metadata: {variance_path.relative_to(REPO_ROOT)}")

## 8. Visualize individual and cumulative explained variance

The scree plot shows the variance explained by individual components. The cumulative curve shows how much total variance is retained as components are added. The horizontal reference lines mark the 80%, 90% and 95% thresholds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show_n = min(30, len(explained))
axes[0].plot(np.arange(1, show_n + 1), explained[:show_n] * 100, marker="o")
axes[0].set_title(f"Scree plot – first {show_n} components")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Explained variance (%)")
axes[0].grid(alpha=0.25)

axes[1].plot(np.arange(1, len(cumulative) + 1), cumulative * 100)
for threshold, n in [(80, n_80), (90, n_90), (95, n_95)]:
    axes[1].axhline(threshold, linestyle="--", alpha=0.8, label=f"{threshold}% ({n} PCs)")
    axes[1].axvline(n, linestyle=":", alpha=0.45)
axes[1].set_title("Cumulative explained variance")
axes[1].set_xlabel("Number of principal components")
axes[1].set_ylabel("Cumulative variance (%)")
axes[1].set_ylim(0, 102)
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "PCA_Explained_Variance.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

## 9. 2D PCA projection

The data is projected onto the first two principal components. Points are colored by `SalePrice` to help inspect whether price variation follows visible gradients in the reduced space.

The first two components usually retain much less than 95% of the total variance. This plot is a two-dimensional visualization, not a substitute for the full PCA representation.

In [ ]:
if X_scaled.shape[1] < 2:
    raise ValueError("At least two retained predictors are required for a 2D PCA visualization.")
pca_2d = PCA(n_components=2, svd_solver="full")
X_pca_2d = pca_2d.fit_transform(X_scaled)

saleprice_for_plot = pd.to_numeric(df.loc[X_scaled.index, "SalePrice"], errors="coerce")
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    X_pca_2d[:, 0], X_pca_2d[:, 1],
    c=saleprice_for_plot, cmap="viridis", alpha=0.68, s=22, edgecolors="none"
)
colorbar = fig.colorbar(scatter, ax=ax)
colorbar.set_label("SalePrice")
ax.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.2f}% variance)")
ax.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.2f}% variance)")
ax.set_title("Ames Housing observations projected onto PC1 and PC2")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "PCA_2D_Projection.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Variance captured by PC1 and PC2 together: {pca_2d.explained_variance_ratio_.sum()*100:.2f}%")

## 10. Interpret principal-component loadings

A loading indicates how strongly an original standardized predictor contributes to a principal component. We inspect the largest absolute loadings for PC1 and PC2. The sign indicates direction; the magnitude indicates contribution strength.

Component labels such as “size and quality” should only be assigned after examining the actual high-loading features in this run.

In [ ]:
loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=kept_features,
    columns=["PC1", "PC2"]
)
loading_rows = []
for component in ["PC1", "PC2"]:
    top = loadings[component].abs().sort_values(ascending=False).head(15).index
    part = loadings.loc[top, component].sort_values(key=lambda s: s.abs(), ascending=True)
    for feature, weight in part.items():
        loading_rows.append({"Component": component, "Feature": feature, "Loading": weight, "Absolute loading": abs(weight)})
loading_summary = pd.DataFrame(loading_rows)
display(loading_summary)
loading_summary.to_csv(OUTPUT_DIR / "week7_top_pca_loadings.csv", index=False)
loadings.to_csv(OUTPUT_DIR / "week7_all_pca_loadings_pc1_pc2.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, component in zip(axes, ["PC1", "PC2"]):
    top = loadings[component].abs().sort_values(ascending=False).head(15).index
    vals = loadings.loc[top, component].sort_values()
    ax.barh(vals.index, vals.values)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Top loadings: {component}")
    ax.set_xlabel("Loading weight")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "PCA_Component_Loadings.png", dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

## 11. Export the PCA-transformed dataset

The final PCA model retains the smallest number of components that reaches at least 95% cumulative explained variance. The transformed dataset includes identifiers and target columns for downstream joining and interpretation, but those columns were not used to fit PCA.

The dashboard-compatible variance table is separately saved as `Week 7/PCA_Explained_Variance.csv`.

In [ ]:
pca_95 = PCA(n_components=n_95, svd_solver="full")
X_pca_95 = pca_95.fit_transform(X_scaled)
pc_columns = [f"PC{i+1}" for i in range(n_95)]
pca_scores = pd.DataFrame(X_pca_95, columns=pc_columns, index=X_scaled.index)

metadata_cols = [c for c in ID_COLS + TARGET_COLS if c in df.columns]
pca_export = pd.concat([
    df.loc[X_scaled.index, metadata_cols].reset_index(drop=True),
    pca_scores.reset_index(drop=True)
], axis=1)
pca_output_path = REPO_ROOT / "Week 7" / "PCA_Transformed_AmesHousing.csv"
pca_export.to_csv(pca_output_path, index=False)

print(f"PCA output shape: {pca_export.shape}")
print(f"Saved PCA scores: {pca_output_path.relative_to(REPO_ROOT)}")
print(f"Saved explained variance: {variance_path.relative_to(REPO_ROOT)}")

## 12. Key findings and reproducibility checks

All findings are generated from the current input file. The final checks confirm that the PCA output contains the required component columns, that the explained-variance file reaches the requested threshold and that selected-feature and PCA exports exist.

In [ ]:
key_findings = [
    f"The Week 6 input contained {len(df):,} observations and {df.shape[1]:,} columns.",
    f"Numeric predictors before filtering: {len(feature_cols):,}.",
    f"Variance threshold retained {len(vt_kept):,} predictors and removed {len(vt_dropped):,}.",
    f"Pairwise correlation pruning retained {len(kept_features):,} predictors and removed {len(dropped_pairs):,} highly correlated predictors at |r| >= {MULTICOLLINEARITY_THRESHOLD:.2f}.",
    f"Combined F-regression and Random Forest ranking selected {len(selected_features):,} original features.",
    f"PCA retained {n_95:,} components for at least 95% variance ({cumulative[n_95-1]*100:.2f}% actual cumulative variance).",
    f"PC1 and PC2 together explain {pca_2d.explained_variance_ratio_.sum()*100:.2f}% of variance.",
]
findings_text = "\n".join(f"{i+1}. {item}" for i, item in enumerate(key_findings))
print(findings_text)
(OUTPUT_DIR / "week7_key_findings.txt").write_text(findings_text, encoding="utf-8")

assert pca_export.shape[0] == X_scaled.shape[0]
assert all(c in pca_export.columns for c in pc_columns)
assert variance_path.exists() and selected_path.exists() and pca_output_path.exists()
assert float(variance_df["Cumulative Explained Variance"].iloc[n_95 - 1]) >= 0.95 - 1e-10
assert not X_scaled.isna().any().any()
print("\nWeek 7 validation checks passed.")

## 13. Conclusion

Week 7 reduces the feature space using two complementary approaches. Feature selection retains a chosen subset of original predictors, while PCA forms a new orthogonal representation of the retained predictor matrix. Standardization is applied before PCA, and explained variance is reported as individual and cumulative values.

**Week 8 dashboard inputs:**
- `Week 7/Selected_Features_AmesHousing.csv`
- `Week 7/PCA_Transformed_AmesHousing.csv`
- `Week 7/PCA_Explained_Variance.csv`

**Supporting evidence:** `Week 7/week7_outputs/` and `Week 7/Figures/`